In [1]:
import numpy as np
import networkx as nx

import fugu
from fugu import Scaffold, Brick
from fugu.bricks import Vector_Input
from fugu.backends import snn_Backend

In [2]:
# Let's take a look at all valid input coding types.
fugu.input_coding_types

['current',
 'unary-B',
 'unary-L',
 'binary-B',
 'binary-L',
 'temporal-B',
 'temporal-L',
 'Raster',
 'Population',
 'Rate',
 'Undefined']

In [3]:
class AND_Operator(Brick):
    def __init__(self, name=None):
        super().__init__()
        self.name = name
        self.is_built = False
        self.metadata = {'D': 1}
        # It doesn't really matter what coding type it is.
        self.supported_codings = fugu.input_coding_types
        # We want the inputs to have two dimensions. 
        self.dimensionality = {'D': 2}

    def build(self, graph, metadata, controlled_nodes, inputs, input_codings):
        if len(inputs) != 2:
            raise ValueError("the AND operator needs exactly two values")

        # The codings of the two inputs should be the same, 
        # so this doesn't really matter and is arbitrary.
        output_codings = [input_codings[0]]

        completed_node_name = self.name + "_complete"
        graph.add_node(completed_node_name, index = -1, threshold = 0.0, decay = 0.0, p = 1.0, potential = 0.0)
        graph.add_edge(controlled_nodes[0]['complete'], completed_node_name, weight = 1.0, delay = 1.0)

        AND_node_name = self.name + "_0"
        # We set the threshold to fire as soon as both inputs send a spike.
        # 1 + 1 = 2, so we go a bit below at 1.9.
        graph.add_node(AND_node_name, index = 0, threshold = 1.9, decay = 1.0, p = 1.0, potential = 0.0)

        graph.add_edge(inputs[0][0], AND_node_name, weight = 1.0, delay = 1.0)
        graph.add_edge(inputs[1][0], AND_node_name, weight = 1.0, delay = 1.0)

        self.is_built = True

        output_lists = [[AND_node_name]]

        return (graph, self.metadata, [{'complete': completed_node_name}], output_lists, output_codings)


In [4]:
scaffold = Scaffold()
scaffold.add_brick(Vector_Input(np.array([1]), coding="Raster", name="input_1"), 'input')
scaffold.add_brick(Vector_Input(np.array([1]), coding="Raster", name="input_2"), 'input')
scaffold.add_brick(AND_Operator(name='AND'), [(0, 0), (1, 0)], output=True)
scaffold.lay_bricks()
scaffold.summary(verbose=1)

backend = snn_Backend()
backend_args = {}
backend_args['record'] = 'all'
backend.compile(scaffold, backend_args)

Scaffold is built: True
-------------------------------------------------------
Bricks:

Brick No.: 0
Brick Tag: input_1-0
Brick Name: input_1
{'tag': 'input_1-0', 'name': 'input_1', 'brick': <fugu.bricks.input_bricks.Vector_Input object at 0x113a086d0>, 'layer': 'input', 'ports': {'output': PortData(spec=PortSpec(name='output', description='', index=0, minimum=1, maximum=1, channels={'data': ChannelSpec(name='data', description='', coding='Raster', shape=(1,), required=True), 'begin': ChannelSpec(name='begin', description='', coding=[], shape=(1,), required=True), 'complete': ChannelSpec(name='complete', description='', coding=[], shape=(1,), required=True)}), channels={'data': ChannelData(spec=ChannelSpec(name='data', description='', coding='Raster', shape=(1,), required=True), neurons=['input_1-0:(0,)']), 'begin': ChannelData(spec=ChannelSpec(name='begin', description='', coding=[], shape=(1,), required=True), neurons=['input_1-0:begin']), 'complete': ChannelData(spec=ChannelSpec(na

In [5]:
scaffold.graph.nodes(data=True)

result = backend.run(10)
print(result)

   time  neuron_number
0   0.0            2.0
1   0.0            5.0
2   0.0            0.0
3   0.0            1.0
4   0.0            3.0
5   0.0            4.0
6   1.0            6.0
7   1.0            7.0


In [6]:
class OR_Operator(Brick):
    def __init__(self, name=None):
        super().__init__()
        self.name = name
        self.is_built = False
        self.metadata = {'D': 1}
        # It doesn't really matter what coding type it is.
        self.supported_codings = fugu.input_coding_types
        # We want the inputs to have two dimensions. 
        self.dimensionality = {'D': 2}

    def build(self, graph, metadata, controlled_nodes, inputs, input_codings):
        if len(inputs) != 2:
            raise ValueError("the AND operator needs exactly two values")

        # The codings of the two inputs should be the same, 
        # so this doesn't really matter and is arbitrary.
        output_codings = [input_codings[0]]

        completed_node_name = self.name + "_complete"
        graph.add_node(completed_node_name, index = -1, threshold = 0.0, decay = 0.0, p = 1.0, potential = 0.0)
        graph.add_edge(controlled_nodes[0]['complete'], completed_node_name, weight = 1.0, delay = 1.0)

        OR_node_name = self.name + "_0"
        # We set the threshold to fire as soon as both inputs send a spike.
        # 1 + 1 = 2, so we go a bit below at 1.9.
        graph.add_node(OR_node_name, index = 0, threshold = 0.9, decay = 1.0, p = 1.0, potential = 0.0)

        graph.add_edge(inputs[0][0], OR_node_name, weight = 1.0, delay = 1.0)
        graph.add_edge(inputs[1][0], OR_node_name, weight = 1.0, delay = 1.0)

        self.is_built = True

        output_lists = [[OR_node_name]]

        return (graph, self.metadata, [{'complete': completed_node_name}], output_lists, output_codings)


In [7]:
scaffold = Scaffold()
scaffold.add_brick(Vector_Input(np.array([1]), coding="Raster", name="input_1"), 'input')
scaffold.add_brick(Vector_Input(np.array([1]), coding="Raster", name="input_2"), 'input')
scaffold.add_brick(OR_Operator(name='OR'), [(0, 0), (1, 0)], output=True)
scaffold.lay_bricks()
scaffold.summary(verbose=1)

backend = snn_Backend()
backend_args = {}
backend_args['record'] = 'all'
backend.compile(scaffold, backend_args)

Scaffold is built: True
-------------------------------------------------------
Bricks:

Brick No.: 0
Brick Tag: input_1-3
Brick Name: input_1
{'tag': 'input_1-3', 'name': 'input_1', 'brick': <fugu.bricks.input_bricks.Vector_Input object at 0x10be43590>, 'layer': 'input', 'ports': {'output': PortData(spec=PortSpec(name='output', description='', index=0, minimum=1, maximum=1, channels={'data': ChannelSpec(name='data', description='', coding='Raster', shape=(1,), required=True), 'begin': ChannelSpec(name='begin', description='', coding=[], shape=(1,), required=True), 'complete': ChannelSpec(name='complete', description='', coding=[], shape=(1,), required=True)}), channels={'data': ChannelData(spec=ChannelSpec(name='data', description='', coding='Raster', shape=(1,), required=True), neurons=['input_1-3:(0,)']), 'begin': ChannelData(spec=ChannelSpec(name='begin', description='', coding=[], shape=(1,), required=True), neurons=['input_1-3:begin']), 'complete': ChannelData(spec=ChannelSpec(na

In [8]:
scaffold.graph.nodes(data=True)

result = backend.run(10)
print(result)

   time  neuron_number
0   0.0            2.0
1   0.0            5.0
2   0.0            0.0
3   0.0            1.0
4   0.0            3.0
5   0.0            4.0
6   1.0            6.0
7   1.0            7.0
